# Strat Pred — Stage 4 Diagnostic

Interactive companion to `gcp/research/strat_engine/strat_pred_diagnose.py`. Same
logic, exposed for ad-hoc exploration. Pulls a saved `metrics_*.json`
from GCS for a (ticker, tf) cell and shows:

- Headline metrics (log-loss, accuracy, ECE) + gate verdict
- Per-bin reliability detail (where ECE concentrates, over/under-confidence)
- Per-class P/R/F1 (does the model predict every class?)
- Confusion matrix + predicted-class distribution (the "never predicted" check)

Set `TICKER`, `TF`, and `CALIBRATION` below, then Run All.

**Default `CALIBRATION = "sigmoid"`** matches the LOCKED production default in
`strat_config.py`. Set to `"isotonic"` to compare a specific older variant, or
`None` to grab whatever was saved most recently (legacy behavior).

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # so gcp.research... imports

from gcp.research.strat_engine.strat_pred_diagnose import (
    load_metrics, list_all_metrics,
    print_summary, print_reliability,
    print_per_class, print_confusion,
)
from gcp.research.strat_engine.strat_config import DEFAULT_CALIBRATION

TICKER = "IWM"
TF = "15m"
CALIBRATION = DEFAULT_CALIBRATION  # "sigmoid" (LOCKED). Override with "isotonic" or None to compare.

## (Optional) List every saved metrics file for this cell

Useful when you've trained multiple calibration variants and want to
see exactly which one will be loaded below.

In [ ]:
rows = list_all_metrics(TICKER, TF)
print(f"{TICKER} {TF} — {len(rows)} metrics file(s) on GCS (oldest → newest):")
print(f"  {'created_at':<27}  {'cal':<9}  {'log-loss':>9}  {'acc':>7}  {'ECE':>7}  verdict")
print("  " + "-" * 78)
for r in rows:
    ll = f"{r['log_loss']:.4f}" if r['log_loss'] is not None else "?"
    ac = f"{r['accuracy']:.3f}" if r['accuracy'] is not None else "?"
    ec = f"{r['ece']:.4f}" if r['ece'] is not None else "?"
    print(f"  {r['ts']:<27}  {r['calibration']:<9}  {ll:>9}  {ac:>7}  {ec:>7}  {r['verdict']}")

## Pull the metrics JSON for the selected calibration

In [ ]:
m = load_metrics(TICKER, TF, calibration=CALIBRATION)
print_summary(m)

## Reliability (where ECE concentrates)

If a bin shows `← UNDERconfident` repeatedly, the calibrator was too
conservative; sigmoid often fixes that vs isotonic. If a single bin is
wildly off and others are fine, that bin is hiding behind the scalar
average — fix that bin specifically.

In [ ]:
print_reliability(m)

## Per-class predictability (does the model predict every class?)

If `1` or `3` shows `← NEVER PREDICTED`, the inside/outside half of the
product is broken even though log-loss/accuracy may pass. This is the
class-imbalance failure mode.

In [ ]:
print_per_class(m)

## Confusion + predicted-class distribution

In [ ]:
print_confusion(m)

## Side-by-side calibration comparison (optional)

Pull both sigmoid + isotonic latest for the same cell and compare ECE +
per-bin reliability. Useful when deciding whether to override the
locked default for a specific (ticker, tf).

In [ ]:
for cal in ("sigmoid", "isotonic"):
    try:
        mm = load_metrics(TICKER, TF, calibration=cal)
        print(f"--- {cal} ---")
        print(f"  log-loss={mm['oos_log_loss']:.4f}  acc={mm['oos_accuracy']:.3f}  "
              f"ECE={mm['ece']:.4f}  verdict={mm['gate']['verdict']}")
    except RuntimeError as e:
        print(f"--- {cal} ---  (no saved run; {e})")

## Optional: reliability curve as a chart

Skip if you don't have matplotlib available; the table above is the
same data.

In [ ]:
try:
    import matplotlib.pyplot as plt
    bins = [b for b in m['ece_bins'] if b['n'] > 0]
    conf = [b['avg_conf'] for b in bins]
    acc  = [b['avg_acc']  for b in bins]
    sizes = [b['n']/m['n_test']*5000 for b in bins]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='perfect calibration')
    ax.scatter(conf, acc, s=sizes, alpha=0.7, label='bins (size=weight)')
    for b in bins:
        ax.annotate(f"n={b['n']}", (b['avg_conf'], b['avg_acc']),
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax.set_xlabel('avg predicted probability')
    ax.set_ylabel('empirical accuracy')
    ax.set_title(f"{TICKER} {TF} ({m.get('calibration','?')}) — ECE={m['ece']:.4f}")
    ax.legend()
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    plt.show()
except ImportError:
    print('matplotlib not available — skipping chart')